# E-commerce Data Engineering Lab

In this notebook I take **50,000 sales orders** and walk them through the 12-step data engineering road-map: **ingest → wrangle → clean → transform → feature-engineer → aggregate → serialize**. It ends with one analytical insight.

**Data used**

| Role | File | Source |
|---|---|---|
| Primary sales orders | `data/raw/sales_records_dirty.csv` (50,250 rows) | The [ExcelBIAnalytics "50000 Sales Records"](https://excelbianalytics.com/wp/downloads-18-sample-csv-files-data-sets-for-testing-sales/) sample file (kept untouched as `data/raw/sales_records_50k_source.csv`), with realistic grime added by [`scripts/add_grime.py`](scripts/add_grime.py) using a fixed seed. |
| Secondary metadata | `data/raw/world_gdp_2014.csv` (222 rows) | Plotly's open [2014 world GDP](https://github.com/plotly/datasets/blob/master/2014_world_gdp_with_codes.csv) dataset: country name, GDP in billions of USD, and ISO country code. |

**Why add grime?** The downloaded file is almost clean. Its only natural problem is trailing spaces in 1,973 country names. Real order exports are messier, so the script adds the kinds of problems a real export would have (a second date format, `$` prices, blanks, spelling variants, a double export). The seed is fixed, so every run finds exactly the same problems. Because I still have the untouched source, Step 9 can check the cleaned result against it.

I use the secondary file for two things: to **enrich** each order with its country's GDP and ISO code, and as the second source for the **Data Dictionary** at the end.

## Setup

All file paths are defined once here. The notebook uses paths relative to the repository root, so it runs the same after a fresh `git clone`.

In [1]:
import gzip
import json
from collections import Counter, defaultdict, namedtuple
from dataclasses import asdict, astuple, dataclass
from datetime import date, datetime
from pathlib import Path

import pandas as pd

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

RAW_SALES = RAW_DIR / "sales_records_dirty.csv"
SOURCE_SALES = RAW_DIR / "sales_records_50k_source.csv"
RAW_GDP = RAW_DIR / "world_gdp_2014.csv"
CLEAN_CSV = PROCESSED_DIR / "sales_clean.csv"
CLEAN_JSON = PROCESSED_DIR / "sales_clean.json.gz"
PROFIT_JSON = PROCESSED_DIR / "profit_by_item_type.json"

## Step 1 — Hello, Data!

I read every column **as text** (`dtype=str`) and turn off pandas' automatic missing-value detection (`keep_default_na=False`). I want to see the file exactly as it was written.

If I let pandas parse it, `Unit Price` would silently become text because of the `$` signs, blank `Units Sold` would become `NaN` and turn the whole column into floats, and the two date formats would be hidden inside generic strings. The grime would be half-hidden before I had even looked at it.

In [2]:
raw_df = pd.read_csv(RAW_SALES, dtype=str, keep_default_na=False)

print(f"Loaded {len(raw_df):,} rows x {raw_df.shape[1]} columns")
raw_df.head(3)

Loaded 50,250 rows x 14 columns


,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
0,Sub-Saharan Africa,Namibia,Household,Offline,M,2015-08-31,897751939,2015-10-12,3604,668.27,502.54,2408445.08,1811154.16,597290.92
1,Europe,Iceland,Baby Food,Online,H,11/20/2010,599480426,1/9/2011,8435,255.28,159.42,2153286.80,1344707.70,808579.10
2,Europe,Russia,Meat,Online,L,6/22/2017,538911855,6/25/2017,4848,$421.89,364.69,2045322.72,1768017.12,277305.60


## Step 2 — Pick the Right Container

| Container | Where I use it | Why |
|---|---|---|
| **Class** (`@dataclass`) | One order: `SalesOrder` | An order must change during cleaning and carry behaviour (`clean()`, `revenue()`, `profit()`, `ship_days()`). A namedtuple is immutable, and a dict cannot hold methods. |
| **namedtuple** | `CountryInfo`, `PriceProfile` | Small, fixed, read-only records. A namedtuple gives named fields (`info.gdp_billions`) with no extra code, and its immutability protects reference data. |
| **dict** | `country_lookup`: country → `CountryInfo`; spelling maps; `region_by_country` | Enrichment means "look up by key" 50,000 times, and a dict does that in constant time. A list search would be 222 times slower per lookup. |
| **set** | Unique countries, unique order IDs, duplicate detection | A set removes duplicates automatically, so `len(set(...))` is the unique count, and membership checks are constant time. |

## Step 3 — Implement Functions and Data Structure

**The functions.** Each helper does one conversion and returns `None` when a value cannot be used. The helpers never guess.

Each one also **returns non-string input unchanged**. That makes cleaning *idempotent*: running `clean()` on an already-clean record changes nothing. Step 7 relies on this to measure the raw and cleaned data with the same code.

In [3]:
DATE_FORMATS = ("%m/%d/%Y", "%Y-%m-%d")          # the source's US format first; ISO seen in the raw file
PRIORITY_CODES = {"C", "H", "M", "L"}
MAX_UNITS = 10_000                               # the largest order line in the source file
UNKNOWN_REGION = "UNKNOWN"


def collapse_spaces(value: str) -> str:
    return " ".join(value.split())


def canonical_spellings(values: pd.Series) -> dict[str, str]:
    """Map each case-insensitive name to its most common spelling in the data."""
    spellings = {}
    for spelling, _ in Counter(values.map(collapse_spaces)).most_common():
        spellings.setdefault(spelling.casefold(), spelling)
    return spellings


def normalize_name(value: str | None, spellings: dict[str, str]) -> str | None:
    """'  south korea ' -> 'South Korea', using the most common spelling as the correct one."""
    if not isinstance(value, str):
        return value
    text = collapse_spaces(value)
    return spellings.get(text.casefold(), text)


def normalize_priority(value: str | None) -> str | None:
    """'High', 'high' or 'h' -> 'H'. Anything that is not C/H/M/L -> None."""
    if not isinstance(value, str):
        return value
    code = value.strip()[:1].upper()
    return code if code in PRIORITY_CODES else None


def parse_price(value: str | float | None) -> float | None:
    """'$421.89' -> 421.89. Returns None when the text is not a number."""
    if not isinstance(value, str):
        return value
    try:
        return float(value.replace("$", "").strip())
    except ValueError:
        return None


def parse_units(value: str | int | None) -> int | None:
    """'3604' -> 3604. Blank or non-numeric text -> None."""
    if not isinstance(value, str):
        return value
    text = value.strip()
    return int(text) if text.isdigit() else None


def parse_date(value: str | date | None) -> date | None:
    """Accepts '8/31/2015' or '2015-08-31'. Anything else -> None."""
    if not isinstance(value, str):
        return value
    for date_format in DATE_FORMATS:
        try:
            return datetime.strptime(value.strip(), date_format).date()
        except ValueError:
            continue
    return None

**Spelling maps.** A country name cannot simply be title-cased: `"cote d'ivoire".title()` gives `Cote D'Ivoire`, and `"bosnia and herzegovina".title()` gives `Bosnia And Herzegovina`. Instead, I compare names case-insensitively and use **the most common spelling in the data** as the correct one. The data votes, and the correct spelling wins by a wide margin because only a small share of rows is misspelled.

In [4]:
COUNTRY_SPELLINGS = canonical_spellings(raw_df["Country"])
ITEM_SPELLINGS = canonical_spellings(raw_df["Item Type"])
CHANNEL_SPELLINGS = canonical_spellings(raw_df["Sales Channel"])

for column, spellings in [("Country", COUNTRY_SPELLINGS), ("Item Type", ITEM_SPELLINGS), ("Sales Channel", CHANNEL_SPELLINGS)]:
    print(f"{column:13} raw spellings: {raw_df[column].nunique():4}  ->  real values: {len(spellings)}")

Country       raw spellings:  768  ->  real values: 185
Item Type     raw spellings:   60  ->  real values: 12
Sales Channel raw spellings:   10  ->  real values: 2


**The data structure.** `SalesOrder` is one order line.

- It starts out holding the raw text.
- `clean()` returns a **new**, typed `SalesOrder` rather than changing itself, so the raw version stays available for before/after comparisons.
- `problems()` lists everything that makes the record unusable.
- `revenue()`, `cost()` and `profit()` are **recalculated** from units and prices. The file's stored totals are not trusted (Step 6 shows why).

`OrderBook` is a collection of orders. It has its own batch-level `clean()`, which also removes duplicates and fills blank regions (that needs *other* rows, so it cannot happen inside a single order), plus revenue and profit totals.

In [5]:
COLUMN_TO_FIELD = {
    "Order ID": "order_id",
    "Order Date": "order_date",
    "Ship Date": "ship_date",
    "Region": "region",
    "Country": "country",
    "Item Type": "item_type",
    "Sales Channel": "sales_channel",
    "Order Priority": "order_priority",
    "Units Sold": "units_sold",
    "Unit Price": "unit_price",
    "Unit Cost": "unit_cost",
}


@dataclass
class SalesOrder:
    """One order line. Holds raw text until clean() returns a typed copy."""

    order_id: str
    order_date: str | date | None
    ship_date: str | date | None
    region: str
    country: str
    item_type: str
    sales_channel: str
    order_priority: str | None
    units_sold: str | int | None
    unit_price: str | float | None
    unit_cost: str | float | None

    @classmethod
    def from_row(cls, row: dict) -> "SalesOrder":
        """Build from one CSV row (a dict). The stored totals are left out on purpose."""
        return cls(**{field: row[column] for column, field in COLUMN_TO_FIELD.items()})

    def clean(self) -> "SalesOrder":
        return SalesOrder(
            order_id=self.order_id.strip(),
            order_date=parse_date(self.order_date),
            ship_date=parse_date(self.ship_date),
            region=self.region.strip(),
            country=normalize_name(self.country, COUNTRY_SPELLINGS),
            item_type=normalize_name(self.item_type, ITEM_SPELLINGS),
            sales_channel=normalize_name(self.sales_channel, CHANNEL_SPELLINGS),
            order_priority=normalize_priority(self.order_priority),
            units_sold=parse_units(self.units_sold),
            unit_price=parse_price(self.unit_price),
            unit_cost=parse_price(self.unit_cost),
        )

    def problems(self) -> list[str]:
        """Reasons this (cleaned) record cannot be used; an empty list means it is valid."""
        issues = []
        if self.order_date is None or self.ship_date is None:
            issues.append("unreadable date")
        elif self.ship_date < self.order_date:
            issues.append("shipped before it was ordered")
        if self.unit_price is None or self.unit_price <= 0:
            issues.append("missing or non-positive price")
        if self.unit_cost is None or self.unit_cost <= 0:
            issues.append("missing or non-positive cost")
        if self.units_sold is None:
            issues.append("missing units")
        elif not 1 <= self.units_sold <= MAX_UNITS:
            issues.append(f"units outside 1-{MAX_UNITS:,}")
        if self.order_priority is None:
            issues.append("unknown priority")
        return issues

    def revenue(self) -> float:
        return round(self.units_sold * self.unit_price, 2)

    def cost(self) -> float:
        return round(self.units_sold * self.unit_cost, 2)

    def profit(self) -> float:
        return round(self.revenue() - self.cost(), 2)

    def ship_days(self) -> int:
        return (self.ship_date - self.order_date).days


class OrderBook:
    """A collection of orders with batch-level cleaning, revenue and profit."""

    def __init__(self, orders: list[SalesOrder]) -> None:
        self.orders = orders

    def __len__(self) -> int:
        return len(self.orders)

    def clean(self) -> tuple["OrderBook", dict]:
        """Clean every record, drop duplicates, fill blank regions, drop invalid rows; return the result and a report."""
        cleaned = [order.clean() for order in self.orders]

        # Deduplicate on the *cleaned* values, so the same order typed two ways still counts once.
        seen, unique = set(), []
        for order in cleaned:
            key = astuple(order)
            if key not in seen:
                seen.add(key)
                unique.append(order)

        # A country always belongs to one region, so the other rows tell us a blank region.
        region_by_country = {order.country: order.region for order in unique if order.region}
        blank_regions = [order for order in unique if not order.region]
        for order in blank_regions:
            order.region = region_by_country.get(order.country, UNKNOWN_REGION)

        valid = [order for order in unique if not order.problems()]
        reasons = Counter(issue for order in unique for issue in order.problems())

        report = {
            "rows before": len(cleaned),
            "exact duplicates removed": len(cleaned) - len(unique),
            "blank regions filled from country": len(blank_regions),
            "invalid rows removed": len(unique) - len(valid),
            "rows after": len(valid),
            **{f"  reason: {reason}": count for reason, count in reasons.items()},
        }
        return OrderBook(valid), report

    def total_revenue(self) -> float:
        return round(sum(order.revenue() for order in self.orders), 2)

    def total_profit(self) -> float:
        return round(sum(order.profit() for order in self.orders), 2)

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame([asdict(order) for order in self.orders])

**Using it.** I populate a list with the first three rows, then clean each one. Row 1's dates are in ISO format and row 3's price has a `$` sign. After cleaning, both become the same types as every other row.

In [6]:
sample = [SalesOrder.from_row(row) for row in raw_df.head(3).to_dict(orient="records")]

for raw in sample:
    clean = raw.clean()
    print(f"{raw.order_id}: raw price {raw.unit_price!r:10} date {raw.order_date!r:13}"
          f"-> {clean.order_date}, {clean.units_sold:,} x {clean.unit_price} = revenue {clean.revenue():,.2f}, profit {clean.profit():,.2f}")

897751939: raw price '668.27'   date '2015-08-31' -> 2015-08-31, 3,604 x 668.27 = revenue 2,408,445.08, profit 597,290.92
599480426: raw price '255.28'   date '11/20/2010' -> 2010-11-20, 8,435 x 255.28 = revenue 2,153,286.80, profit 808,579.10
538911855: raw price '$421.89'  date '6/22/2017'  -> 2017-06-22, 4,848 x 421.89 = revenue 2,045,322.72, profit 277,305.60


## Step 4 — Bulk Loaded

Now I load everything. `DataFrame.to_dict(orient="records")` turns each row into a dict, and each dict becomes a `SalesOrder` in an `OrderBook`.

The GDP file is mapped into a **dict**: `country → CountryInfo`. Before trusting a plain dict, I check that no country appears twice in the file. Otherwise the dict would silently keep whichever row came last.

In [7]:
CountryInfo = namedtuple("CountryInfo", ["code", "gdp_billions"])

raw_book = OrderBook([SalesOrder.from_row(row) for row in raw_df.to_dict(orient="records")])

gdp_df = pd.read_csv(RAW_GDP)
assert gdp_df["COUNTRY"].is_unique, "a duplicated country would be silently overwritten in the dict"
country_lookup: dict[str, CountryInfo] = {
    row["COUNTRY"]: CountryInfo(row["CODE"], row["GDP (BILLIONS)"]) for row in gdp_df.to_dict(orient="records")
}

print(f"OrderBook: {len(raw_book):,} orders")
print(f"country_lookup: {len(country_lookup)} countries")
print("country_lookup['Germany'] =", country_lookup["Germany"])

OrderBook: 50,250 orders
country_lookup: 222 countries
country_lookup['Germany'] = CountryInfo(code='DEU', gdp_billions=3820.0)


## Step 5 — Quick Profiling

This is a first look at the numbers **before** any cleaning, taking the file at face value. `PriceProfile` is a namedtuple, so the result reads like a small report.

In [8]:
PriceProfile = namedtuple("PriceProfile", ["min", "mean", "max", "not_a_number"])


def profile_prices(raw_prices: pd.Series) -> PriceProfile:
    numbers = pd.to_numeric(raw_prices, errors="coerce")   # '$421.89' cannot be read -> NaN
    return PriceProfile(
        min=float(numbers.min()),
        mean=round(float(numbers.mean()), 2),
        max=float(numbers.max()),
        not_a_number=int(numbers.isna().sum()),
    )


price_profile = profile_prices(raw_df["Unit Price"])
raw_countries = set(raw_df["Country"])
raw_order_ids = set(raw_df["Order ID"])

print(price_profile)
print(f"Rows: {len(raw_df):,}   unique Order IDs: {len(raw_order_ids):,}")
print(f"Unique country values:  {len(raw_countries)}")
print(f"  after normalize_name: {len({normalize_name(c, COUNTRY_SPELLINGS) for c in raw_countries})}")
print(f"Unique priority values: {sorted(set(raw_df['Order Priority']))}")

PriceProfile(min=-668.27, mean=265.14, max=668.27, not_a_number=902)
Rows: 50,250   unique Order IDs: 50,000
Unique country values:  768
  after normalize_name: 185
Unique priority values: ['C', 'Critical', 'H', 'High', 'L', 'Low', 'M', 'Medium', 'c', 'critical', 'h', 'high', 'l', 'low', 'm', 'medium']


The profile already points at four problems:

- **The minimum price is negative.** A product cannot cost less than nothing.
- **Some prices are not numbers at all** (`not_a_number`), so the mean is being calculated on fewer rows than the file contains.
- **There are more rows than Order IDs**, so some orders appear more than once.
- **The country set is inflated.** The raw values contain several hundred "countries", which collapse to 185 once the spelling is normalised. Priority should be four one-letter codes but arrives in many forms.

## Step 6 — Spot the Grime

I turn each suspicion into a check that counts the affected rows and shows one example. I use `repr()` for the examples so that stray spaces are visible.

In [9]:
def find_grime(df: pd.DataFrame) -> pd.DataFrame:
    """Count each kind of dirty value in the raw, all-text sales file."""
    units = pd.to_numeric(df["Units Sold"], errors="coerce")
    price = pd.to_numeric(df["Unit Price"].str.replace("$", "", regex=False), errors="coerce")
    stored_revenue = pd.to_numeric(df["Total Revenue"], errors="coerce")
    order_date = df["Order Date"].map(parse_date)
    ship_date = df["Ship Date"].map(parse_date)

    def misspelled(column: str, spellings: dict[str, str]) -> pd.Series:
        return df[column].ne(df[column].map(lambda value: normalize_name(value, spellings)))

    checks = {
        "exact duplicate row":                   ("Order ID", df.duplicated()),
        "price stored with a $ sign":            ("Unit Price", df["Unit Price"].str.startswith("$")),
        "negative price":                        ("Unit Price", df["Unit Price"].str.startswith("-")),
        "blank units sold":                      ("Units Sold", df["Units Sold"].str.strip().eq("")),
        f"units above {MAX_UNITS:,}":            ("Units Sold", units.gt(MAX_UNITS)),
        "blank region":                          ("Region", df["Region"].str.strip().eq("")),
        "date in ISO instead of M/D/YYYY":       ("Order Date", df["Order Date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")),
        "ship date before order date":           ("Ship Date", ship_date.lt(order_date)),
        "country with odd case or spacing":      ("Country", misspelled("Country", COUNTRY_SPELLINGS)),
        "item type with odd case or spacing":    ("Item Type", misspelled("Item Type", ITEM_SPELLINGS)),
        "sales channel with odd case or spacing": ("Sales Channel", misspelled("Sales Channel", CHANNEL_SPELLINGS)),
        "priority not a C/H/M/L code":           ("Order Priority", ~df["Order Priority"].isin(PRIORITY_CODES)),
        "blank stored Total Revenue":            ("Total Revenue", df["Total Revenue"].str.strip().eq("")),
        "stored Total Revenue != units x price": ("Total Revenue", (stored_revenue - units * price).abs().gt(0.01)),
    }
    rows = []
    for issue, (column, mask) in checks.items():
        example = repr(df.loc[mask, column].iloc[0]) if mask.any() else ""
        rows.append({"issue": issue, "column": column, "rows affected": int(mask.sum()), "example": example})
    return pd.DataFrame(rows)


grime = find_grime(raw_df)
grime

,issue,column,rows affected,example
0,exact duplicate row,Order ID,250,'647978627'
1,price stored with a $ sign,Unit Price,902,'$421.89'
2,negative price,Unit Price,41,'-9.33'
3,blank units sold,Units Sold,120,''
4,"units above 10,000",Units Sold,25,'99999'
5,blank region,Region,152,''
6,date in ISO instead of M/D/YYYY,Order Date,2518,'2015-08-31'
7,ship date before order date,Ship Date,59,'12/29/2015'
8,country with odd case or spacing,Country,3135,'Moldova '
9,item type with odd case or spacing,Item Type,605,'FRUITS'


The **country with odd case or spacing** count includes the **1,973 trailing spaces that were already in the downloaded file** (e.g. `'Samoa '`), not just the injected variants. Grime is not only a classroom invention.

The secondary file has grime of its own that matters for the join: **the two sources name some countries differently.**

In [10]:
our_countries = {normalize_name(country, COUNTRY_SPELLINGS) for country in raw_countries}
not_in_gdp = sorted(our_countries - set(country_lookup))

print(f"Countries in the sales data: {len(our_countries)}")
print(f"...with no exact match in the GDP file: {len(not_in_gdp)}")
not_in_gdp

Countries in the sales data: 185
...with no exact match in the GDP file: 13


['Cape Verde',
 'Democratic Republic of the Congo',
 'East Timor',
 'Federated States of Micronesia',
 'Myanmar',
 'Nauru',
 'North Korea',
 'Republic of the Congo',
 'South Korea',
 'The Bahamas',
 'The Gambia',
 'United States of America',
 'Vatican City']

**What each problem would break if I ignored it:**

1. **Duplicates** would double-count revenue and profit for those orders.
2. **`$` prices** cannot be read as numbers. **Negative prices** would subtract revenue. Neither is a real sale price.
3. **Blank and 99,999 unit counts** mean I cannot know what was actually sold. 99,999 is ten times the largest real order.
4. **Mixed date formats** break any date arithmetic, and a **ship date before the order date** is impossible. One of the two dates is wrong, and I cannot tell which.
5. **Spelling variants** of countries, item types, channels and priorities split one real value into several groups, which corrupts every "per country" or "per item" result.
6. **Blank regions** would drop orders out of every regional total.
7. **Stored totals that are blank or 10× too high** show that the file's `Total Revenue` column cannot be trusted. Totals must be recalculated from units × price.
8. **Country names that differ between the two files** would silently leave those orders without GDP after the join (Step 8 shows this).

## Step 7 — Cleaning Rules

Every rule lives inside `SalesOrder.clean()` (per record) and `OrderBook.clean()` (per batch):

| Problem | Rule | Why |
|---|---|---|
| `$` prices | Strip the `$`, then convert to a number | The value is correct; only the formatting is wrong. |
| Negative prices | **Drop the row** | I cannot tell a refund from a typo, and guessing would invent revenue. |
| Blank units, or units above 10,000 | **Drop the row** | Without a real quantity there is no real revenue figure. |
| ISO dates | Parse both formats into one `date` type | Every date is readable once the format is known. |
| Ship date before order date | **Drop the row** | One of the two dates is wrong and I cannot tell which, so the shipping time is unknown. |
| Blank region | **Fill it** from the region of the same country in other rows | Every country belongs to exactly one region in this data, so the answer is known, not guessed. |
| Country, item type and channel spelling | Compare case-insensitively and use the most common spelling | `"  south korea"` and `"SOUTH KOREA"` are the same country. |
| Priority variants | Take the first letter, upper-cased: `High`, `high` and `h` all become `H` | They all mean the same priority. |
| Stored totals | **Ignore them** and recalculate revenue, cost and profit from units and unit prices | The stored column is sometimes blank or wrong, while units × price is always checkable. |
| Exact duplicates | Keep the first copy | This looks like an order exported twice. |

In [11]:
clean_book, cleaning_report = raw_book.clean()
pd.Series(cleaning_report, name="count").to_frame()

,count
rows before,50250
exact duplicates removed,250
blank regions filled from country,150
invalid rows removed,243
rows after,49757
"reason: units outside 1-10,000",25
reason: missing units,120
reason: shipped before it was ordered,58
reason: missing or non-positive price,40


The report counts reasons per row, and a row can fail for more than one reason. That is why the reasons can add up to more than the number of rows removed.

**Before vs after.** Because `clean()` is idempotent, the same function measures both books:

In [12]:
def quality_counts(book: OrderBook) -> dict[str, int]:
    records = book.orders
    return {
        "rows": len(records),
        "duplicate order_ids": len(records) - len({order.order_id for order in records}),
        "distinct country spellings": len({order.country for order in records}),
        "distinct item type spellings": len({order.item_type for order in records}),
        "distinct sales channel values": len({order.sales_channel for order in records}),
        "distinct priority values": len({order.order_priority for order in records}),
        "blank regions": sum(1 for order in records if not order.region.strip()),
        "rows that fail validation": sum(1 for order in records if order.clean().problems()),
    }


before_after = pd.DataFrame({"before": quality_counts(raw_book), "after": quality_counts(clean_book)})
before_after

,before,after
rows,50250,49757
duplicate order_ids,250,0
distinct country spellings,768,185
distinct item type spellings,60,12
distinct sales channel values,10,2
distinct priority values,16,4
blank regions,152,0
rows that fail validation,245,0


## Step 8 — Transformations

### 8a. Priority code → label and rank

The file stores priority as a one-letter code. For reading and sorting, I map it to a **label** and a numeric **rank** (1 = most urgent). The table shows every raw spelling and what it becomes:

In [13]:
PRIORITY_LABELS = {"C": "Critical", "H": "High", "M": "Medium", "L": "Low"}
PRIORITY_RANK = {"C": 1, "H": 2, "M": 3, "L": 4}

priority_map = raw_df["Order Priority"].value_counts().rename_axis("raw value").reset_index(name="rows")
priority_map["normalized"] = priority_map["raw value"].map(normalize_priority)
priority_map["label"] = priority_map["normalized"].map(PRIORITY_LABELS)
priority_map["rank"] = priority_map["normalized"].map(PRIORITY_RANK)
priority_map

,raw value,rows,normalized,label,rank
0,L,12472,L,Low,4
1,H,12368,H,High,2
2,M,12356,M,Medium,3
3,C,12351,C,Critical,1
4,Medium,73,M,Medium,3
5,l,69,L,Low,4
6,Low,63,L,Low,4
7,medium,61,M,Medium,3
8,c,58,C,Critical,1
9,critical,58,C,Critical,1


### 8b. Objects → typed DataFrame

For column-wise work I convert the `OrderBook` back into a DataFrame. Each `SalesOrder` becomes a row through `dataclasses.asdict`. The dates become real `datetime64` columns, and I add the priority label and rank from 8a.

In [14]:
orders = clean_book.to_dataframe()
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["ship_date"] = pd.to_datetime(orders["ship_date"])
orders["priority_label"] = orders["order_priority"].map(PRIORITY_LABELS)
orders["priority_rank"] = orders["order_priority"].map(PRIORITY_RANK)

orders.dtypes

order_id                    str
order_date        datetime64[s]
ship_date         datetime64[s]
region                      str
country                     str
item_type                   str
sales_channel               str
order_priority              str
units_sold                int64
unit_price              float64
unit_cost               float64
priority_label              str
priority_rank             int64
dtype: object

### 8c. Enrichment: joining the GDP data

First, here is what a naive join by country name does:

In [15]:
naive_join = orders.merge(gdp_df, left_on="country", right_on="COUNTRY", how="left")
unmatched = naive_join["GDP (BILLIONS)"].isna()

print(f"orders before join: {len(orders):,}   after naive join: {len(naive_join):,}")
print(f"orders left without GDP: {unmatched.sum():,} ({unmatched.mean():.1%}) across {naive_join.loc[unmatched, 'country'].nunique()} countries")

orders before join: 49,757   after naive join: 49,757
orders left without GDP: 3,596 (7.2%) across 13 countries


The row count is safe (each country appears once in the GDP file), but the join **silently lost data**. The two sources spell some countries differently, so thousands of orders get no GDP. Any "revenue relative to GDP" result would quietly leave those countries out.

The fix is an explicit **alias table** from the sales spelling to the GDP spelling. Each country is then resolved to exactly one match, and a `country_match` column records how:

- `exact`: the names already agree.
- `alias`: matched through the alias table.
- `not found`: the GDP file has no row for it (Nauru and Vatican City). Those orders are kept, with blank GDP.

Then I join with `validate="many_to_one"`, which makes pandas raise an error if any country ever matches more than once.

In [16]:
COUNTRY_ALIASES = {
    "Cape Verde": "Cabo Verde",
    "Democratic Republic of the Congo": "Congo, Democratic Republic of the",
    "East Timor": "Timor-Leste",
    "Federated States of Micronesia": "Micronesia, Federated States of",
    "Myanmar": "Burma",
    "North Korea": "Korea, North",
    "Republic of the Congo": "Congo, Republic of the",
    "South Korea": "Korea, South",
    "The Bahamas": "Bahamas, The",
    "The Gambia": "Gambia, The",
    "United States of America": "United States",
}


def resolve_country(name: str) -> tuple[CountryInfo | None, str]:
    if name in country_lookup:
        return country_lookup[name], "exact"
    if COUNTRY_ALIASES.get(name) in country_lookup:
        return country_lookup[COUNTRY_ALIASES[name]], "alias"
    return None, "not found"


country_rows = []
for country in sorted(set(orders["country"])):
    info, match = resolve_country(country)
    country_rows.append({"country": country, **(info._asdict() if info else {}), "country_match": match})

country_table = pd.DataFrame(country_rows).rename(columns={"code": "country_code", "gdp_billions": "country_gdp_billions"})
orders = orders.merge(country_table, on="country", how="left", validate="many_to_one")

print(f"orders after safe join: {len(orders):,}")
print(orders.drop_duplicates("country")["country_match"].value_counts().to_string())
country_table[country_table["country_match"] != "exact"]

orders after safe join: 49,757
country_match
exact        172
alias         11
not found      2


,country,country_code,country_gdp_billions,country_match
27,Cape Verde,CPV,1.98,alias
38,Democratic Republic of the Congo,COD,32.67,alias
43,East Timor,TLS,4.51,alias
50,Federated States of Micronesia,FSM,0.34,alias
110,Myanmar,MMR,65.29,alias
112,Nauru,NaN,NaN,not found
119,North Korea,PRK,28.00,alias
130,Republic of the Congo,COG,14.11,alias
151,South Korea,KOR,1410.00,alias
164,The Bahamas,BHM,8.65,alias


## Step 9 — Feature Engineering

| New column | How it is calculated |
|---|---|
| `revenue` | `units_sold × unit_price`: the same formula as `SalesOrder.revenue()` |
| `cost` | `units_sold × unit_cost` |
| `profit` | `revenue − cost` |
| `profit_margin` | `profit ÷ revenue` |
| `ship_days` | Days from order date to ship date |
| `days_since_order` | Days from the order date to the **snapshot date** |
| `order_year`, `order_month`, `order_weekday` | Calendar parts of the order date |
| `is_online` | `True` when the sales channel is Online |

**Snapshot date.** I measure `days_since_order` from the day after the last order in the data, not from today. With `date.today()`, every re-run of this notebook would produce different numbers, so the instructor's run would not match mine.

In [17]:
SNAPSHOT_DATE = orders["order_date"].max() + pd.Timedelta(days=1)

orders["revenue"] = (orders["units_sold"] * orders["unit_price"]).round(2)
orders["cost"] = (orders["units_sold"] * orders["unit_cost"]).round(2)
orders["profit"] = (orders["revenue"] - orders["cost"]).round(2)
orders["profit_margin"] = (orders["profit"] / orders["revenue"]).round(4)
orders["ship_days"] = (orders["ship_date"] - orders["order_date"]).dt.days
orders["days_since_order"] = (SNAPSHOT_DATE - orders["order_date"]).dt.days
orders["order_year"] = orders["order_date"].dt.year
orders["order_month"] = orders["order_date"].dt.strftime("%Y-%m")
orders["order_weekday"] = orders["order_date"].dt.day_name()
orders["is_online"] = orders["sales_channel"].eq("Online")

# The vectorised columns and the class methods must agree, or one of them is wrong.
assert abs(orders["revenue"].sum() - clean_book.total_revenue()) < 1
assert abs(orders["profit"].sum() - clean_book.total_profit()) < 1

print(f"Snapshot date: {SNAPSHOT_DATE.date()}")
orders[["order_id", "order_date", "ship_date", "units_sold", "unit_price", "revenue",
        "profit", "profit_margin", "ship_days", "days_since_order", "is_online"]].head()

Snapshot date: 2017-07-29

,order_id,order_date,ship_date,units_sold,unit_price,revenue,profit,profit_margin,ship_days,days_since_order,is_online
0,897751939,2015-08-31,2015-10-12,3604,668.27,2408445.08,597290.92,0.2480,42,698,False
1,599480426,2010-11-20,2011-01-09,8435,255.28,2153286.80,808579.10,0.3755,50,2443,True
2,538911855,2017-06-22,2017-06-25,4848,421.89,2045322.72,277305.60,0.1356,3,37,True
3,459845054,2012-02-28,2012-03-20,7225,421.89,3048155.25,413270.00,0.1356,21,1978,True
4,626391351,2010-08-12,2010-09-13,1975,205.70,406257.50,174965.25,0.4307,32,2543,True


**Checking the pipeline against the untouched source.** Because I kept the original download, I can test whether cleaning recovered the true values. For every order that survived cleaning, I compare my recalculated revenue and profit, and my cleaned country and item type, with the original file's values for the same Order ID.

In [18]:
source = pd.read_csv(SOURCE_SALES, dtype={"Order ID": str})
source["Country"] = source["Country"].str.strip()
check = orders.merge(source, left_on="order_id", right_on="Order ID", how="left", validate="one_to_one")

pd.Series({
    "orders compared": len(check),
    "revenue differs from source Total Revenue": int((check["revenue"] - check["Total Revenue"]).abs().gt(0.01).sum()),
    "profit differs from source Total Profit": int((check["profit"] - check["Total Profit"]).abs().gt(0.01).sum()),
    "country differs from source": int(check["country"].ne(check["Country"]).sum()),
    "item type differs from source": int(check["item_type"].ne(check["Item Type"]).sum()),
    "priority differs from source": int(check["order_priority"].ne(check["Order Priority"]).sum()),
}, name="count").to_frame()

,count
orders compared,49757
revenue differs from source Total Revenue,0
profit differs from source Total Profit,0
country differs from source,0
item type differs from source,0
priority differs from source,0


Every surviving order now matches the original file exactly. The cleaning rules restored the true values rather than inventing new ones, and the rows I dropped were the ones that could not be recovered.

## Step 10 — Mini-Aggregation

**Profit per item type**, calculated two ways. The first uses a plain dict built from the `SalesOrder` objects. The second uses `pandas.groupby`, which also adds revenue, units and the profit margin.

In [19]:
profit_dict: dict[str, float] = defaultdict(float)
for order in clean_book.orders:
    profit_dict[order.item_type] += order.profit()

by_item = (
    orders.groupby("item_type")
    .agg(orders=("order_id", "count"),
         units=("units_sold", "sum"),
         revenue=("revenue", "sum"),
         profit=("profit", "sum"))
    .sort_values("profit", ascending=False)
)
by_item["profit_margin"] = by_item["profit"] / by_item["revenue"]
by_item["share_of_profit"] = by_item["profit"] / by_item["profit"].sum()

assert all(abs(profit_dict[item] - value) < 1 for item, value in by_item["profit"].items())

print(f"Total revenue: ${clean_book.total_revenue():,.2f}   total profit: ${clean_book.total_profit():,.2f}   from {len(clean_book):,} orders")
by_item.round({"revenue": 0, "profit": 0, "profit_margin": 3, "share_of_profit": 3})

Total revenue: $65,881,947,396.45   total profit: $19,439,464,995.08   from 49,757 orders


,orders,units,revenue,profit,profit_margin,share_of_profit
item_type,,,,,,
Cosmetics,4178,20871047,9.124822e+09,3.628849e+09,0.398,0.187
Household,4121,20438647,1.365853e+10,3.387297e+09,0.248,0.174
Office Supplies,4123,20580161,1.340201e+10,2.598245e+09,0.194,0.134
Baby Food,4061,20193025,5.154875e+09,1.935703e+09,0.376,0.100
Cereal,4126,20513411,4.219609e+09,1.817283e+09,0.431,0.093
Clothes,4135,20619017,2.253246e+09,1.514261e+09,0.672,0.078
Vegetables,4166,20813577,3.206540e+09,1.313961e+09,0.410,0.068
Meat,4191,20791130,8.771570e+09,1.189253e+09,0.136,0.061
Snacks,4144,20756803,3.167073e+09,1.144530e+09,0.361,0.059


Two more views help explain the result: **price and cost per unit** for each item type, and **shipping time by priority**.

In [20]:
unit_economics = (
    orders.groupby("item_type")
    .agg(unit_price=("unit_price", "first"), unit_cost=("unit_cost", "first"))
    .assign(profit_per_unit=lambda df: df["unit_price"] - df["unit_cost"])
    .sort_values("profit_per_unit", ascending=False)
)
unit_economics

,unit_price,unit_cost,profit_per_unit
item_type,,,
Cosmetics,437.20,263.33,173.87
Household,668.27,502.54,165.73
Office Supplies,651.21,524.96,126.25
Baby Food,255.28,159.42,95.86
Cereal,205.70,117.11,88.59
Clothes,109.28,35.84,73.44
Vegetables,154.06,90.93,63.13
Meat,421.89,364.69,57.20
Snacks,152.58,97.44,55.14


In [21]:
(
    orders.groupby(["priority_rank", "priority_label"])
    .agg(orders=("order_id", "count"), avg_ship_days=("ship_days", "mean"), median_ship_days=("ship_days", "median"))
    .round(2)
)

,,orders,avg_ship_days,median_ship_days
priority_rank,priority_label,,,
1,Critical,12385,24.97,25.0
2,High,12418,24.96,25.0
3,Medium,12430,25.17,25.0
4,Low,12524,24.89,25.0


### Insight

**Revenue is a poor stand-in for profit.**

- **Household** and **Office Supplies** bring in the most revenue ($13.7B and $13.4B), but **Cosmetics** earns the most profit ($3.6B, 18.7% of all profit) on only $9.1B of revenue. Its margin is 39.8%, against 24.8% and 19.4%.
- **Meat** is the 4th-largest item by revenue (13.3% of all revenue) but only 8th by profit (6.1%), because it keeps just 13.6% of each sale. **Clothes** is the opposite: 3.4% of revenue but 7.8% of profit, with a 67.2% margin.
- The reason is **profit per unit**, not sales volume. Every item type sells almost the same amount (4,061–4,191 orders and 20.2–21.3 million units each), so the ranking is set entirely by the gap between unit price and unit cost. Cosmetics keeps $173.87 per unit and Fruits just $2.41.

**A second finding: order priority has no effect on shipping.** Critical orders take 24.97 days on average and Low-priority orders 24.89 days, with a median of 25 days for every priority. In a real business, that would mean the priority flag is being ignored in the warehouse.

**Two caveats:**

1. The source is a sample file built for testing, not real company data. The equal volumes, fixed prices and priority-blind shipping are traits of how it was generated, so the findings describe this dataset rather than a real market.
2. The grime I added does not change these results. Step 9 shows that every order kept after cleaning matches the original file, and the 243 rows that could not be recovered (0.5%) are spread across all item types.


## Step 11 — Serialization Checkpoint

I save the cleaned, enriched data in two formats:

- **CSV** for spreadsheets and quick inspection.
- **JSON** (`orient="records"`, one object per order) for web apps and APIs. With about 50,000 orders the plain JSON would be about 27 MB, because every record repeats all 26 field names. This repository keeps committed files under 20 MB, so I write it **gzip-compressed** (`.json.gz`). pandas compresses and decompresses automatically from the file extension.

Dates are written as plain `YYYY-MM-DD` text so both formats agree. The per-item-type profit dict is saved as its own small JSON file.

Then I **read both files back** and check that the row counts, revenue and profit survived the round trip.

In [22]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

export = orders.assign(
    order_date=orders["order_date"].dt.strftime("%Y-%m-%d"),
    ship_date=orders["ship_date"].dt.strftime("%Y-%m-%d"),
)
export.to_csv(CLEAN_CSV, index=False)
export.to_json(CLEAN_JSON, orient="records")
PROFIT_JSON.write_text(json.dumps({item: round(value, 2) for item, value in sorted(profit_dict.items())}, indent=2))

csv_back, json_back = pd.read_csv(CLEAN_CSV), pd.read_json(CLEAN_JSON)
round_trip = pd.DataFrame({
    "rows": [len(orders), len(csv_back), len(json_back)],
    "revenue": [orders["revenue"].sum(), csv_back["revenue"].sum(), json_back["revenue"].sum()],
    "profit": [orders["profit"].sum(), csv_back["profit"].sum(), json_back["profit"].sum()],
}, index=["in memory", CLEAN_CSV.name, CLEAN_JSON.name]).round(2)

for path in (CLEAN_CSV, CLEAN_JSON, PROFIT_JSON):
    print(f"{path}  ({path.stat().st_size / 1024 / 1024:.1f} MB)")
round_trip

data/processed/sales_clean.csv  (9.0 MB)
data/processed/sales_clean.json.gz  (3.5 MB)
data/processed/profit_by_item_type.json  (0.0 MB)


,rows,revenue,profit
in memory,49757,6.588195e+10,1.943946e+10
sales_clean.csv,49757,6.588195e+10,1.943946e+10
sales_clean.json.gz,49757,6.588195e+10,1.943946e+10


In [23]:
with gzip.open(CLEAN_JSON, "rt") as file:
    print(json.dumps(json.load(file)[0], indent=2))

{
  "order_id": "897751939",
  "order_date": "2015-08-31",
  "ship_date": "2015-10-12",
  "region": "Sub-Saharan Africa",
  "country": "Namibia",
  "item_type": "Household",
  "sales_channel": "Offline",
  "order_priority": "M",
  "units_sold": 3604,
  "unit_price": 668.27,
  "unit_cost": 502.54,
  "priority_label": "Medium",
  "priority_rank": 3,
  "country_code": "NAM",
  "country_gdp_billions": 13.11,
  "country_match": "exact",
  "revenue": 2408445.08,
  "cost": 1811154.16,
  "profit": 597290.92,
  "profit_margin": 0.248,
  "ship_days": 42,
  "days_since_order": 698,
  "order_year": 2015,
  "order_month": "2015-08",
  "order_weekday": "Monday",
  "is_online": false
}
